# Lesson 11a: Model-Based Reinforcement Learning Theory

**Learning Objectives:**
- Understand the distinction between model-free and model-based RL
- Learn dynamics model representations (deterministic vs stochastic)
- Master planning algorithms (shooting methods, trajectory optimization)
- Study Dyna architecture and integrated planning/learning
- Explore uncertainty quantification in learned models
- Understand World Models and latent space planning

**Prerequisites:** Dynamic Programming (2a), DQN (7a), Continuous Control (10a)

**References:**
- Sutton & Barto Chapter 8: Planning and Learning
- Deisenroth et al. (2011) - PILCO
- Chua et al. (2018) - PETS
- Janner et al. (2019) - MBPO
- Ha & Schmidhuber (2018) - World Models

## 1. Introduction to Model-Based RL

### 1.1 Model-Free vs Model-Based

**Model-Free RL:**
- Learn value function or policy directly from experience
- No explicit representation of environment dynamics
- Examples: Q-Learning, SARSA, DQN, PPO, SAC
- **Pros**: Simple, no model errors, works in complex environments
- **Cons**: Sample inefficient, poor generalization, no planning

**Model-Based RL:**
- Learn model of environment: $\hat{p}(s'|s,a)$ and $\hat{r}(s,a)$
- Use model for planning or to generate synthetic data
- Examples: Dyna-Q, PILCO, PETS, MBPO, MuZero
- **Pros**: Sample efficient, can reason about future, transfer learning
- **Cons**: Model errors compound, computational cost, complex to implement

### 1.2 The Model-Based RL Pipeline

```
1. Collect data: (s, a, r, s') from environment
2. Learn dynamics: Train f̂(s,a) → s', r̂(s,a) → r
3. Planning: Use model to optimize policy
   - Option A: Plan directly (MPC, trajectory optimization)
   - Option B: Generate synthetic experience for model-free RL
4. Execute: Apply policy in real environment
5. Repeat: Collect more data, improve model
```

### 1.3 When to Use Model-Based RL

**Good for:**
- Sample-limited domains (robotics, expensive simulations)
- Structured environments (physics-based, continuous dynamics)
- When generalization across tasks is needed
- Multi-task learning and transfer

**Challenging for:**
- High-dimensional state spaces (raw images)
- Stochastic/chaotic dynamics
- Partial observability
- Long-horizon tasks (model errors accumulate)

## 2. Dynamics Models

### 2.1 Deterministic Models

**Forward dynamics:**
$$s_{t+1} = f(s_t, a_t)$$

**Neural network parameterization:**
$$\hat{s}_{t+1} = f_\theta(s_t, a_t)$$

**Training objective** (supervised learning):
$$\mathcal{L}(\theta) = \mathbb{E}_{(s,a,s') \sim \mathcal{D}}[\|f_\theta(s,a) - s'\|^2]$$

**Delta model** (often more stable):
$$\Delta s = f_\theta(s,a)$$
$$\hat{s}_{t+1} = s_t + \Delta s$$

### 2.2 Stochastic Models

**Probabilistic dynamics:**
$$p(s_{t+1}|s_t, a_t)$$

**Gaussian model:**
$$p(s_{t+1}|s_t, a_t) = \mathcal{N}(\mu_\theta(s_t, a_t), \Sigma_\theta(s_t, a_t))$$

**Network outputs:**
- Mean: $\mu_\theta(s,a) \in \mathbb{R}^{d_s}$
- Covariance: Diagonal $\Sigma = \text{diag}(\sigma_1^2, ..., \sigma_{d_s}^2)$ (common)
- Or full covariance matrix (expensive!)

**Training objective** (negative log-likelihood):
$$\mathcal{L}(\theta) = -\mathbb{E}[\log p_\theta(s'|s,a)]$$
$$= \mathbb{E}\left[\frac{\|s' - \mu_\theta(s,a)\|^2_{\Sigma_\theta^{-1}}}{2} + \frac{1}{2}\log|\Sigma_\theta| + \text{const}\right]$$

### 2.3 Ensemble Models

**Bootstrap ensemble:**
- Train $K$ independent models on different data subsets
- Diversity through bootstrap sampling

**Prediction:**
$$\hat{s}_{t+1}^{(k)} = f_{\theta_k}(s_t, a_t), \quad k=1,...,K$$

**Mean prediction:**
$$\bar{s}_{t+1} = \frac{1}{K}\sum_{k=1}^K \hat{s}_{t+1}^{(k)}$$

**Uncertainty estimate** (epistemic uncertainty):
$$\text{Var}[s_{t+1}] = \frac{1}{K}\sum_{k=1}^K (\hat{s}_{t+1}^{(k)} - \bar{s}_{t+1})^2$$

**Benefits:**
- Uncertainty quantification (crucial for model-based RL!)
- Better generalization
- Robustness to model errors

### 2.4 Reward Models

**Separate reward model:**
$$\hat{r}_t = r_\phi(s_t, a_t)$$

Or:
$$\hat{r}_t = r_\phi(s_t, a_t, s_{t+1})$$

**Training:**
$$\mathcal{L}(\phi) = \mathbb{E}[(r - r_\phi(s,a))^2]$$

**Note**: Often simpler than dynamics (scalar vs vector)

## 3. Planning with Learned Models

### 3.1 Model Predictive Control (MPC)

**Core idea**: Optimize action sequence over finite horizon, execute first action, replan

**Optimization problem:**
$$a^*_{0:H-1} = \arg\max_{a_{0:H-1}} \sum_{t=0}^{H-1} \hat{r}(s_t, a_t)$$
subject to: $s_{t+1} = \hat{f}(s_t, a_t)$, $s_0 = s_{\text{current}}$

**Algorithm:**
```
At each timestep:
1. Solve optimization for action sequence [a_0, ..., a_{H-1}]
2. Execute first action a_0 in environment
3. Observe new state
4. Repeat (replan from new state)
```

**Receding horizon**: Only execute first action, then replan
- Corrects for model errors!
- Computational cost: must optimize at every step

### 3.2 Random Shooting

**Simple MPC method:**
1. Sample $N$ random action sequences: $\{a^{(i)}_{0:H-1}\}_{i=1}^N$
2. Rollout each sequence using model: $s_{t+1} = \hat{f}(s_t, a_t)$
3. Compute returns: $R^{(i)} = \sum_{t=0}^{H-1} \hat{r}(s_t^{(i)}, a_t^{(i)})$
4. Select best sequence: $i^* = \arg\max_i R^{(i)}$
5. Execute $a_0^{(i^*)}$

**Pros**: Simple, parallelizable
**Cons**: Inefficient for long horizons or high-dimensional actions

### 3.3 Cross-Entropy Method (CEM)

**Iterative refinement of action distribution:**

Initialize: $p(a_{0:H-1}) = \mathcal{N}(\mu_0, \Sigma_0)$

For iteration $k = 1, ..., K$:
1. Sample $N$ action sequences from $p(a_{0:H-1})$
2. Evaluate returns using model
3. Select top $M$ elite sequences
4. Fit new distribution to elite sequences:
   $$\mu_{k+1} = \frac{1}{M}\sum_{i \in \text{elite}} a^{(i)}$$
   $$\Sigma_{k+1} = \frac{1}{M}\sum_{i \in \text{elite}} (a^{(i)} - \mu_{k+1})(a^{(i)} - \mu_{k+1})^T$$

Return: $a_0^* = \mu_K[0]$ (first action of final mean)

**Pros**: More sample efficient than random shooting
**Cons**: Can get stuck in local optima

### 3.4 Trajectory Optimization

**Gradient-based optimization:**

Objective:
$$J(a_{0:H-1}) = \sum_{t=0}^{H-1} r(s_t, a_t)$$

Constraints:
$$s_{t+1} = f(s_t, a_t)$$

**Gradient computation** (backpropagation through time):
$$\nabla_{a_t} J = \nabla_{a_t} r_t + \nabla_{a_t} s_{t+1}^T \nabla_{s_{t+1}} J_{t+1:H-1}$$

Can use automatic differentiation!

**Differential Dynamic Programming (DDP):**
- Second-order method (uses Hessian)
- Very efficient for continuous control
- Used in robotics (e.g., iLQR variant)

## 4. Dyna Architecture

### 4.1 The Dyna Framework

**Sutton (1990)**: Integrate planning and learning

**Key idea**: Learn model from real experience, use model to generate synthetic experience for learning

```
Loop:
  (a) Act: Execute action in environment
  (b) Learn: Update value function/policy from real experience
  (c) Model: Update environment model
  (d) Plan: Generate synthetic experience, update value function/policy
```

### 4.2 Dyna-Q Algorithm

**Tabular Q-learning + planning:**

```
Initialize Q(s,a), Model(s,a) for all s, a

Loop:
  # (a) Direct RL
  Take action a in state s, observe r, s'
  Q(s,a) ← Q(s,a) + α[r + γ max_a' Q(s',a') - Q(s,a)]
  
  # (c) Model learning
  Model(s,a) ← (r, s')  # Store observed transition
  
  # (d) Planning (n simulated steps)
  Repeat n times:
    s_sim ← random previously observed state
    a_sim ← random action taken from s_sim
    r_sim, s'_sim ← Model(s_sim, a_sim)
    Q(s_sim, a_sim) ← Q(s_sim, a_sim) + α[r_sim + γ max_a Q(s'_sim, a) - Q(s_sim, a_sim)]
```

**Benefits:**
- $n$ planning steps per real step → sample efficiency
- Combines direct and indirect RL

### 4.3 Dyna-Q+ (Exploration Bonus)

**Problem**: Model may become stale in non-stationary environments

**Solution**: Exploration bonus for transitions not seen recently

$$r^+ = r + \kappa\sqrt{\tau}$$

where $\tau$ = time since $(s,a)$ was tried in real environment, $\kappa$ = bonus weight

Encourages revisiting states to update model!

### 4.4 Prioritized Sweeping

**Idea**: Focus planning on states where updates would be largest

**Priority:**
$$p(s,a) = |r + \gamma \max_{a'} Q(s', a') - Q(s,a)|$$

**Algorithm:**
- Maintain priority queue of state-action pairs
- Plan for high-priority pairs first
- When Q(s,a) updated, update priorities of predecessor states

**Much more efficient than uniform random planning!**

## 5. Modern Model-Based Deep RL

### 5.1 PETS (Probabilistic Ensembles with Trajectory Sampling)

**Chua et al. (2018)**

**Key components:**
1. **Bootstrap ensemble** of probabilistic neural networks
2. **Uncertainty-aware planning** (CEM with trajectory sampling)
3. **Online data collection** and model updates

**Model:**
- Ensemble of $K$ networks: $\{f_{\theta_k}\}_{k=1}^K$
- Each predicts Gaussian: $f_{\theta_k}(s,a) = (\mu_k(s,a), \sigma_k^2(s,a))$

**Planning (CEM):**
```
For each trajectory sample:
  1. Randomly select model k ~ Uniform(1, K)
  2. Sample transition: s' ~ N(μ_k(s,a), σ_k²(s,a))
  3. Compute reward r(s, a)
  4. Repeat for H steps
  
Select action sequence with highest return
```

**Uncertainty propagation:** Different models + stochastic sampling → uncertainty-aware planning

### 5.2 MBPO (Model-Based Policy Optimization)

**Janner et al. (2019)**

**Idea**: Use model to generate short synthetic rollouts, train model-free RL on mixed data

**Key insight**: Short rollouts → less model error accumulation

**Algorithm:**
```
1. Initialize policy π and dynamics model f̂
2. Collect real data with π
3. Train f̂ on real data
4. For k rollout steps:
     Sample state s from real buffer
     Generate k-step rollout: s, a ~ π(s), s' ~ f̂(s,a), r, ...
     Add synthetic transitions to model buffer
5. Train π with SAC on mixture of real + model buffers
6. Repeat from step 2
```

**Branched rollouts:**
- Start from real states (reduces compounding errors)
- Short horizons ($k = 1-5$ typically)
- Combine with strong model-free algorithm (SAC)

**State-of-the-art results** on MuJoCo benchmarks!

### 5.3 MuZero

**Schrittwieser et al. (2020)**

**Breakthrough**: Learn latent model (not raw state dynamics)

**Three learned functions:**
1. **Representation**: $h^0 = h_\theta(o_{1:t})$ (history → latent state)
2. **Dynamics**: $h^{k+1} = g_\theta(h^k, a^k)$ (latent transition)
3. **Prediction**: $p^k, v^k = f_\theta(h^k)$ (policy, value)

**Key idea**: Don't predict observations, predict values and policies!

**Planning**: MCTS in learned latent space

**Results**: Matches AlphaZero on board games, works on Atari (no game rules needed!)

## 6. Uncertainty and Exploration

### 6.1 Types of Uncertainty

**Aleatoric (irreducible) uncertainty:**
- Inherent stochasticity in environment
- Cannot be reduced with more data
- Example: Die roll outcome

**Epistemic (reducible) uncertainty:**
- Uncertainty due to limited data
- Reduced by collecting more samples
- Example: Unvisited state-action pairs

### 6.2 Quantifying Model Uncertainty

**Ensemble disagreement:**
$$\sigma^2(s,a) = \frac{1}{K}\sum_{k=1}^K \|\mu_k(s,a) - \bar{\mu}(s,a)\|^2$$

High disagreement → epistemic uncertainty → explore!

**Bayesian Neural Networks:**
- Posterior over weights: $p(\theta|\mathcal{D})$
- Prediction: $p(s'|s,a,\mathcal{D}) = \int p(s'|s,a,\theta)p(\theta|\mathcal{D})d\theta$
- Variance = uncertainty

**Dropout as Bayesian approximation:**
- MC Dropout: Multiple forward passes with dropout
- Variance across predictions ≈ uncertainty

### 6.3 Optimism in the Face of Uncertainty

**Upper Confidence Bound (UCB) for planning:**

$$Q_{\text{UCB}}(s,a) = \hat{Q}(s,a) + \beta \cdot \sigma_{\text{model}}(s,a)$$

- $\hat{Q}(s,a)$: Expected return
- $\sigma_{\text{model}}(s,a)$: Model uncertainty
- $\beta$: Exploration coefficient

Encourage exploration of uncertain states!

### 6.4 Safe Exploration

**Problem**: Model errors can lead to dangerous actions

**Pessimistic planning:**
$$Q_{\text{LCB}}(s,a) = \hat{Q}(s,a) - \beta \cdot \sigma_{\text{model}}(s,a)$$

Avoid uncertain states (safety-critical applications)

**Ensemble robustness:**
- Plan for worst-case model: $\min_k Q_k(s,a)$
- Guarantees safety across model ensemble

## 7. World Models

### 7.1 Learning in Latent Space

**Ha & Schmidhuber (2018)**

**Motivation**: High-dimensional observations (images) hard to model

**Solution**: Learn compressed latent representation

**Architecture:**
```
Observation o_t → VAE Encoder → Latent z_t
                ↓
         VAE Decoder → Reconstruction ô_t

Latent dynamics: z_{t+1} = MDN-RNN(z_t, a_t, h_t)
Policy: a_t = π(z_t)
```

**Components:**
1. **VAE (Vision)**: $o \to z$ (compress)
2. **MDN-RNN (Memory)**: $p(z_{t+1}|z_t, a_t)$ (dynamics)
3. **Controller**: $a = C(z)$ (policy, often simple linear)

### 7.2 Training World Models

**Stage 1: Learn vision model**
$$\mathcal{L}_{VAE} = \mathbb{E}[\|o - \hat{o}\|^2] + \text{KL}[q(z|o) \| p(z)]$$

**Stage 2: Learn dynamics in latent space**
$$\mathcal{L}_{RNN} = -\mathbb{E}[\log p(z_{t+1}|z_t, a_t, h_t)]$$

**Stage 3: Train policy**
- Train entirely in dream (latent space)
- Evolution strategies or policy gradient
- No real environment interaction!

**Benefits:**
- Train in parallel (no need for real environment)
- Very sample efficient
- Can learn from videos!

### 7.3 Dreamer

**Hafner et al. (2020, 2021)**

**Improvements over World Models:**
- **Recurrent State Space Model (RSSM):**
  - Deterministic path: $h_t = f(h_{t-1}, z_{t-1}, a_{t-1})$
  - Stochastic state: $z_t \sim p(z_t|h_t)$
  - Observation: $o_t \sim p(o_t|h_t, z_t)$

- **Actor-critic in latent space:**
  - Value function: $V_\psi(z_t)$
  - Policy: $a_t \sim \pi_\theta(a_t|z_t)$

- **Gradient-based policy learning** (not evolution)

**State-of-the-art on visual control tasks!**

## 8. Theoretical Analysis

### 8.1 Compounding Model Errors

**Problem**: Errors accumulate over long horizons

Suppose model error at each step: $\epsilon = \|s' - \hat{s}'\|$

After $H$ steps: $\epsilon_H \approx H \epsilon$ (linear accumulation)

In practice often worse: $\epsilon_H \approx \epsilon e^{\lambda H}$ (exponential!)

**Solutions:**
1. **Short rollouts** (MBPO approach)
2. **Ensemble robustness** (PETS approach)
3. **Frequent replanning** (MPC approach)
4. **Latent models** (MuZero, Dreamer)

### 8.2 When Does Model-Based RL Help?

**Ross & Bagnell (2012)** analysis:

Model-based better when:
$$\epsilon_{\text{model}}^2 \ll \epsilon_{\text{model-free}}^2 / N_{\text{samples}}$$

Where:
- $\epsilon_{\text{model}}$: Model error
- $\epsilon_{\text{model-free}}$: Policy gradient variance
- $N_{\text{samples}}$: Number of samples

**Intuition**: Model-based helps when variance reduction outweighs model bias

### 8.3 Sample Complexity

**Model-free** (e.g., PPO):
$$N_{\text{samples}} \approx O\left(\frac{|\mathcal{S}||\mathcal{A}|}{\epsilon^2}\right)$$

**Model-based** (e.g., Dyna):
$$N_{\text{samples}} \approx O\left(\frac{|\mathcal{S}||\mathcal{A}|}{n\epsilon^2}\right)$$

where $n$ = planning steps per real step

**Speedup**: $n \times$ (in tabular case)

In practice: 2-10× sample efficiency on continuous control tasks

## 9. Practical Considerations

### 9.1 Data Collection Strategy

**Random exploration** (initial):
- Collect diverse data
- Cover state space
- Typically 1000-5000 random steps

**On-policy collection:**
- Collect with current policy
- Update model
- Improve policy

**Active learning:**
- Select actions to reduce model uncertainty
- Query informative state-action pairs

### 9.2 Model Architecture Choices

**Feedforward networks:**
```python
state, action → [256, 256] → Δstate, reward
```
- Good for: Markovian systems
- Fast training and inference

**Recurrent networks (RNN/LSTM):**
```python
state, action, hidden → LSTM → Δstate, reward, hidden
```
- Good for: Partial observability
- Captures temporal dependencies

**Latent models (VAE-based):**
```python
observation → VAE encoder → latent
latent, action → dynamics → next latent
```
- Good for: High-dimensional observations (images)
- Compact representation

### 9.3 Hyperparameters

**Model training:**
- Learning rate: $3 \times 10^{-4}$
- Batch size: 256-512
- Ensemble size: 5-7
- Training epochs: Until validation loss plateaus

**Planning:**
- Horizon: 5-30 (shorter for complex dynamics)
- CEM iterations: 5-10
- Population size: 200-500
- Elite fraction: 0.1

**MBPO-specific:**
- Rollout length: 1-5 steps
- Model:real ratio: 0.05-0.5 (more model data)

### 9.4 Debugging Tips

**Check model accuracy:**
- Visualize predictions vs ground truth
- Track validation loss over time
- Monitor ensemble disagreement

**Validate planning:**
- Compare model-predicted returns with real returns
- Visualize planned trajectories
- Check if actions are reasonable

**Common issues:**
- Model overfitting → more data, regularization
- Compounding errors → shorter rollouts, replanning
- Poor exploration → add uncertainty bonuses
- Expensive planning → reduce horizon, use GPU

## Summary

### Key Takeaways

1. **Model-Based RL** learns environment dynamics and uses them for planning or data generation

2. **Dynamics models** can be deterministic, stochastic, or ensemble-based:
   - Ensembles provide uncertainty quantification (critical!)

3. **Planning methods**:
   - Random shooting (simple)
   - CEM (more efficient)
   - MPC (replan at every step)
   - Gradient-based (iLQR, DDP)

4. **Dyna architecture** integrates planning and learning:
   - Learn model from real experience
   - Generate synthetic experience for policy learning

5. **Modern deep MBRL**:
   - PETS: Uncertainty-aware planning
   - MBPO: Short model rollouts + model-free RL
   - MuZero: Latent space planning
   - Dreamer: World model with gradient-based learning

6. **Uncertainty** is crucial:
   - Epistemic uncertainty → explore
   - Ensemble disagreement → quantify uncertainty

7. **Model errors compound** over long horizons:
   - Use short rollouts
   - Frequent replanning
   - Ensemble robustness

8. **Sample efficiency**: 2-10× improvement over model-free in continuous control

### Next Steps

- **Lesson 11b**: Implement Dyna-Q, PETS-style MPC, MBPO
- **Lesson 12**: Exploration strategies (curiosity, count-based)
- **Lesson 13**: Multi-agent RL

### Further Reading

1. Sutton & Barto (2018) - Chapter 8: Planning and Learning with Tabular Methods
2. Deisenroth et al. (2011) - "PILCO: A Model-Based and Data-Efficient Approach to Policy Search"
3. Chua et al. (2018) - "Deep Reinforcement Learning in a Handful of Trials using Probabilistic Dynamics Models" (PETS)
4. Janner et al. (2019) - "When to Trust Your Model: Model-Based Policy Optimization" (MBPO)
5. Schrittwieser et al. (2020) - "Mastering Atari, Go, Chess and Shogi by Planning with a Learned Model" (MuZero)
6. Ha & Schmidhuber (2018) - "World Models"
7. Hafner et al. (2020) - "Dream to Control: Learning Behaviors by Latent Imagination" (Dreamer)

## Exercises

### Theoretical Questions

1. **Derive** the gradient for trajectory optimization with a learned dynamics model

2. **Prove** that Dyna-Q converges to optimal Q-values given a perfect model

3. **Analyze** how model error $\epsilon$ compounds over horizon $H$

4. **Compare** sample complexity of model-free vs model-based RL

5. **Explain** why ensemble disagreement measures epistemic uncertainty

### Computational Exercises

6. **Implement** Dyna-Q on a gridworld and compare with Q-learning

7. **Build** an ensemble of dynamics models and visualize uncertainty

8. **Compare** random shooting vs CEM for planning

9. **Implement** prioritized sweeping and measure speedup over Dyna-Q

10. **Train** a latent dynamics model (VAE + RNN) on CartPole

### Practical Applications

11. **Implement** simple MPC with CEM for continuous control

12. **Build** PETS-style planning with probabilistic ensembles

13. **Implement** MBPO with short rollouts + SAC

14. **Analyze** how rollout length affects MBPO performance

15. **Compare** model-based vs model-free sample efficiency on MuJoCo